# Heart Signal Reconstruction & Joint TPE Hyperparameter Optimization Pipeline

This notebook documents, optimizes, visualizes, and evaluates the end-to-end signal processing and machine learning pipeline to reconstruct the cardiac signal from a **5-dimensional wrist-worn sensor recording** (`[ppg1, ppg2, accx, accy, accz]`) and estimate the Heart Rate in **Beats Per Minute (BPM)** on each individual 8-second window ($N = 1000$ samples at $f_s = 125\text{ Hz}$).

---

## 1. Signal Processing Pipeline

Each 5-dimensional window $\mathbf{X}_w \in \mathbb{R}^{5 \times N}$ is processed **individually** in a **single forward pass** from $n = 0$ to $N - 1$:

```mermaid
flowchart LR
    A["5D Window Input<br/>[ppg1, ppg2, accx, accy, accz]"] --> B["Stage 1: prepare_channels<br/>+ 4th Channel acc_mag<br/>Zero-Phase Bandpass (0.5-4.0 Hz)<br/>+ Unit-Variance Norm"]
    B --> C1["Primary d1[n] (PPG1)"]
    B --> C2["Primary d2[n] (PPG2)"]
    B --> D["Stage 2: 4M-Tap MISO Regressor U<br/>[accx, accy, accz, acc_mag]"]
    C1 & D --> E1["Stage 3a: Single-Pass LMS Branch 1<br/>e1[n] = d1[n] - w1^T[n] a[n]"]
    C2 & D --> E2["Stage 3b: Single-Pass LMS Branch 2<br/>e2[n] = d2[n] - w2^T[n] a[n]"]
    E1 & E2 & D --> F["Stage 4: Motion-Suppressed<br/>SNR-Weighted Late Fusion<br/>e_fused[n] = w1*e1 + w2*e2"]
    F & D --> G["Stage 5: HR Estimation<br/>Motion-Suppressed Harmonic Periodogram<br/>or Elgendi Systolic Peaks"]
```

### Stage 1: Preprocessing & 4th Non-Linear Reference Channel (`preprocessing.py`)
1. **4th Accelerometer Reference (`acc_mag`)**:
   Before filtering, the instantaneous Euclidean norm of the raw triaxial accelerometer is computed to capture total motion intensity and non-linear inter-axis coupling:
   $$a_{\text{mag}}[n] = \sqrt{a_x^2[n] + a_y^2[n] + a_z^2[n]}$$
   forming 6 channels: `[ppg1, ppg2, accx, accy, accz, acc_mag]`.
2. **Zero-Phase Bandpass Filtering ($30\text{–}240\text{ BPM} \equiv 0.5\text{–}4.0\text{ Hz}$)**:
   A 3rd-order zero-phase Butterworth filter (`sosfiltfilt` with odd reflection padding) eliminates startup step-response transients and preserves exact phase alignment between PPG and accelerometer channels.
3. **Amplitude Normalization**:
   Each channel is normalized to zero mean and unit standard deviation along the time axis, yielding `ppg_norm` of shape `(2, N)` and `acc_norm` of shape `(4, N)`.

### Stage 2: 4-Channel MISO Reference Signal Construction (`reference_construction.py`)
The $K = 4$ normalized reference channels (`accx, accy, accz, acc_mag`) are expanded into tapped delay-line vectors of order $M \in [8, 24]$ ($4M \in [32, 96]$ total taps):
$$\hat{m}[n] = \sum_{k=1}^{4} \mathbf{w}_k^T[n]\mathbf{a}_k[n] = \mathbf{w}^T[n]\mathbf{a}[n], \qquad \mathbf{a}[n] \in \mathbb{R}^{4M}$$

### Stage 3: Single-Pass Adaptive MISO LMS Filtering (`LMS_algorithms.py`)
Starting from $\mathbf{w}[0] = \mathbf{0}$, each filter adapts online sample-by-sample in a **single forward pass** ($n = 0, \dots, N-1$):
$$e[n] = d[n] - \hat{m}[n] = s[n] + m[n] - \mathbf{w}^T[n]\mathbf{a}[n]$$
- **Standard LMS (`standard_lms`)**: $M \in [8, 24],\; \mu \in [5 \times 10^{-4}, 0.02]$
- **Normalized LMS (`nlms`)**: $M \in [8, 24],\; \mu \in [0.05, 0.8],\; \delta \in [10^{-4}, 10^{-1}]$
- **Variable Step-Size LMS (`vs_lms`)**: $M \in [8, 24],\; \mu_{\max} \in [0.005, 0.08],\; \mu_{\min} \in [10^{-4}, 0.005],\; \alpha \in [0.90, 0.995],\; \gamma \in [10^{-4}, 10^{-2}]$

### Stage 4: Motion-Suppressed SNR-Weighted Late Fusion (`fusion.py`)
To prevent a narrowband residual motion peak from inflating a branch's SNR weight, each branch's normalized PSD $P_{e_i}(f)$ is penalized by the mean normalized PSD of the 4 accelerometer channels $P_{\text{acc}}(f)$ in $[0.5, 4.0]\text{ Hz}$:
$$S_i(f) = P_{e_i}(f) \cdot \max\!\left(0.05,\; 1.0 - 0.6 \, P_{\text{acc}}(f)\right), \qquad i \in \{1, 2\}$$
$\text{SNR}_i$ is computed around the dominant peak of $S_i(f)$ (plus its 2nd harmonic), and $e_{\text{fused}}[n] = w_1 e_1[n] + w_2 e_2[n]$.

### Stage 5: Window-Level Heart Rate Estimation (`bpm_detection.py`)
1. **Motion-Suppressed Harmonic Periodogram (`motion_suppressed_periodogram_bpm`)**:
   Applies soft spectral subtraction against $P_{\text{acc}}(f)$ and reinforces candidate fundamentals $f$ supported by a 2nd cardiac harmonic at $2f$:
   $$\text{score}(f) = P_e(f) \cdot \max\!\left(0.05,\; 1 - \alpha_{\text{sup}} P_{\text{acc}}(f)\right) \cdot \left(1 + w_{\text{harm}} P_e(2f)\right)$$
2. **Systolic Peak Detection — Elgendi's Algorithm (`elgendi_systolic_peak_bpm`)**:
   Uses two moving averages ($\text{MA}_{\text{peak}}$ over $W_1$ and $\text{MA}_{\text{beat}}$ over $W_2$) with dynamic threshold offset $\beta \cdot \overline{S}$ to extract systolic peaks.

---

## 2. Outlier-Penalized Composite Objective & 2-Phase TPE Optimization (`optimization.py`)

$$\mathcal{L}(\hat{\mathbf{y}}, \mathbf{y}) = \text{AAE} + 0.35 \cdot \text{RMSE} + 0.50 \cdot \frac{1}{W}\sum_{w=1}^{W} \max\!\left(0,\, |\hat{y}_w - y_w| - 10.0\right)$$

In [ ]:
import sys
from pathlib import Path
from pprint import pprint
import numpy as np
import matplotlib.pyplot as plt

# Ensure project root is on sys.path whether notebook is run from root or restructured_implementation/
current_dir = Path.cwd().resolve()
project_root = current_dir.parent if current_dir.name == "restructured_implementation" else current_dir
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from restructured_implementation import (
    FS,
    WIN,
    STEP,
    load_ts,
    get_participant_groups,
    bandpass,
    normalize_amplitude,
    prepare_channels,
    lms_bucket,
    motion_suppressed_periodogram_bpm,
    optimize_hr_pipeline,
    objective_function,
)

## 3. Load Training and Test Datasets

We load `IEEEPPG_TRAIN.ts` and `IEEEPPG_TEST.ts` as individual 8-second windows of shape `(5, 1000)` and identify the recording subject IDs.

In [ ]:
train_path = project_root / "IEEEPPG_TRAIN.ts"
test_path = project_root / "IEEEPPG_TEST.ts"

X_train, y_train = load_ts(str(train_path))
X_test, y_test = load_ts(str(test_path))

train_groups = get_participant_groups(X_train)
test_groups = get_participant_groups(X_test)

print(f"Training set : X_train = {X_train.shape}, y_train = {y_train.shape} | Subjects = {len(np.unique(train_groups))}")
print(f"Test set     : X_test  = {X_test.shape}, y_test  = {y_test.shape} | Subjects = {len(np.unique(test_groups))}")

## 4. Joint 2-Phase TPE Hyperparameter Optimization & Algorithm Comparison

We execute `optimize_hr_pipeline` on `(X_train, y_train)` to obtain `best_model` and `best_hyperparameters`, and compare all evaluated LMS (`standard_lms`, `nlms`, `vs_lms`) and HR detection (`periodogram`, `elgendi`) variants.

In [ ]:
optimization_results = optimize_hr_pipeline(
    X=X_train,
    y=y_train,
    n_splits=5,
    phase1_trials=25,
    phase1_sample_size=120,
    phase1_top_k=6,
    phase2_trials=30,
    prune_percentile=50.0,
    seed=42,
)

best_model = optimization_results["best_model"]
best_hyperparameters = optimization_results["best_hyperparameters"]
best_cv_score = optimization_results["best_score"]
comparison = optimization_results["comparison_summary"]

print("=== Winning Conditional Tree Hyperparameters ===")
pprint(best_hyperparameters)
print(f"\nBest Cross-Validated Composite Objective Score (Train): {best_cv_score:.3f}")

In [ ]:
# Display table and bar charts comparing all evaluated LMS and HR algorithms
phase1_by_lms = comparison["phase1_by_lms"]
phase2_by_pair = comparison["phase2_by_pair"]

print("=== Phase 1: Best Motion-Suppressed Spectral SNR Proxy by LMS Variant ===")
for lms_name, entry in sorted(phase1_by_lms.items()):
    print(f"  {lms_name:14s} | Mean Spectral SNR = {entry['mean_snr']:.4f} | Params = {entry['lms_params']}")

print("\n=== Phase 2: Best Cross-Validation Performance by (LMS + HR Detector) Pair ===")
print(f"{'Algorithm Combination':30s} | {'Objective Score':>15s} | {'CV AAE [BPM]':>13s} | {'Folds':>5s}")
print("-" * 72)
for pair_name, entry in sorted(phase2_by_pair.items(), key=lambda kv: kv[1]["score_objective"]):
    print(f"{pair_name:30s} | {entry['score_objective']:15.3f} | {entry['score_aae']:13.3f} | {entry['folds_evaluated']:5d}")

fig, axes = plt.subplots(1, 2, figsize=(14, 4.8))

# Left plot: Phase 1 Spectral SNR by LMS algorithm
lms_names = list(phase1_by_lms.keys())
lms_snrs = [phase1_by_lms[k]["mean_snr"] for k in lms_names]
axes[0].bar(lms_names, lms_snrs, color=["tab:blue", "tab:orange", "tab:green"][:len(lms_names)], edgecolor="black", alpha=0.85)
axes[0].set_title("Phase 1 Screening: Motion-Suppressed SNR Proxy by LMS Variant")
axes[0].set_ylabel("Mean In-Band Cardiac SNR (Higher is Better)")
axes[0].grid(axis="y", alpha=0.3)

# Right plot: Phase 2 Cross-Validated Objective Score & AAE across Algorithm Pairs
pair_labels = list(phase2_by_pair.keys())
obj_vals = [phase2_by_pair[k]["score_objective"] for k in pair_labels]
aae_vals = [phase2_by_pair[k]["score_aae"] for k in pair_labels]
x_pos = np.arange(len(pair_labels))
width = 0.38

axes[1].bar(x_pos - width / 2, obj_vals, width, label="Composite Objective Score", color="tab:red", edgecolor="black", alpha=0.85)
axes[1].bar(x_pos + width / 2, aae_vals, width, label="Mean Absolute Error (AAE [BPM])", color="tab:blue", edgecolor="black", alpha=0.85)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(pair_labels, rotation=25, ha="right")
axes[1].set_title("Phase 2 Cross-Validation: Compared Algorithm Combinations")
axes[1].set_ylabel("Error Score (Lower is Better)")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Visual Walkthrough: Single-Window Step-by-Step Signal Transformation Using the Optimized Pipeline

Using the optimal `best_hyperparameters` found in Section 4, we inspect how a single 8-second window (`sample_idx = 0`) transforms across all 5 stages:
1. **Subplot Row 1**: Raw 5D sensor signals (`ppg1, ppg2`, `accx, accy, accz`) + computed 4th non-linear reference channel `acc_mag`.
2. **Subplot Row 2**: Zero-phase 3rd-order Bandpass Filtered ($0.5\text{–}4.0\text{ Hz}$) signals (eliminating startup transient ringing).
3. **Subplot Row 3**: Unit-variance normalized primary inputs $d_1[n], d_2[n]$ and 4-channel reference $\mathbf{a}[n]$ (`accx, accy, accz, acc_mag`).
4. **Subplot Row 4**: Single-pass MISO LMS adaptation from $\mathbf{w}[0] = \mathbf{0}$, producing motion estimates $\hat{m}_1[n], \hat{m}_2[n]$ and cleaned signals $e_1[n], e_2[n]$.
5. **Subplot Row 5**: Motion-Suppressed SNR-Weighted Late Fusion waveform $e_{\text{fused}}[n]$ (left) and Motion-Suppressed Harmonic Periodogram spectrum vs. Accelerometer spectrum (right), highlighting the Estimated BPM vs. Ground-Truth BPM.

In [ ]:
sample_idx = 0
raw_win = X_train[sample_idx]          # Shape: (5, 1000)
true_bpm_sample = y_train[sample_idx]
t_axis = np.arange(WIN) / FS           # Time axis in seconds (0 to 8 s)

# Stage 1: Construct 4th channel acc_mag, zero-phase bandpass (order=3, causal=False), and normalize
raw_acc_mag = np.sqrt(raw_win[2] ** 2 + raw_win[3] ** 2 + raw_win[4] ** 2)
raw_6ch = np.vstack([raw_win, raw_acc_mag[np.newaxis, :]])
bp_6ch = bandpass(raw_6ch, lo=0.5, hi=4.0, fs=FS, order=3, causal=False)
ppg_norm_sample, acc_norm_sample = prepare_channels(raw_win, fs=FS, lo=0.5, hi=4.0, order=3, causal=False)
d1, d2 = ppg_norm_sample[0], ppg_norm_sample[1]

# Stages 2, 3 & 4: Single-pass MISO LMS + Motion-Suppressed Late Fusion using best_hyperparameters
opt_lms_algo = best_hyperparameters["lms_algorithm"]
opt_lms_params = dict(best_hyperparameters["lms_params"])
opt_M = int(opt_lms_params.pop("M"))

e_fused_sample, sample_details = lms_bucket(
    d1,
    d2,
    acc_norm_sample,
    algorithm=opt_lms_algo,
    M=opt_M,
    return_details=True,
    **opt_lms_params,
)
e1_sample = sample_details["e1"]
e2_sample = sample_details["e2"]
m_hat1_sample = sample_details["m_hat1"]
m_hat2_sample = sample_details["m_hat2"]
f_info = sample_details["fusion_info"]

# Stage 5: Motion-Suppressed Harmonic Periodogram Spectrum
hr_params_spec = (
    dict(best_hyperparameters["hr_params"])
    if best_hyperparameters["hr_algorithm"] == "periodogram"
    else {"nfft": 16384, "window_type": "hamming", "suppression_alpha": 0.6, "harmonic_weight": 0.35}
)
est_bpm_sample, spec_info = motion_suppressed_periodogram_bpm(
    e_fused_sample,
    acc_norm=acc_norm_sample,
    fs=FS,
    return_spectrum=True,
    **hr_params_spec,
)

fig, axes = plt.subplots(5, 2, figsize=(15, 15))

# Row 0: Raw 5D Signals + 4th Channel acc_mag
axes[0, 0].plot(t_axis, raw_win[0], label="Raw PPG1", color="tab:red", alpha=0.85)
axes[0, 0].plot(t_axis, raw_win[1], label="Raw PPG2", color="tab:orange", alpha=0.85)
axes[0, 0].set_title("Subplot 1a: Raw Dual PPG Channels")
axes[0, 0].legend(loc="upper right", fontsize=8); axes[0, 0].grid(alpha=0.3)

for k, (lbl, col) in enumerate(zip(["Acc X", "Acc Y", "Acc Z", "Acc Mag (4th Ch)"], ["tab:blue", "tab:green", "tab:purple", "black"])):
    axes[0, 1].plot(t_axis, raw_6ch[2 + k], label=lbl, color=col, lw=1.3 if k == 3 else 1.0, alpha=0.85)
axes[0, 1].set_title("Subplot 1b: Raw 3-Axis Accelerometer + 4th Channel (acc_mag)")
axes[0, 1].legend(loc="upper right", fontsize=8); axes[0, 1].grid(alpha=0.3)

# Row 1: Zero-Phase Bandpass Filtered (0.5 - 4.0 Hz, order=3, causal=False)
axes[1, 0].plot(t_axis, bp_6ch[0], label="Zero-Phase BP PPG1", color="tab:red", alpha=0.85)
axes[1, 0].plot(t_axis, bp_6ch[1], label="Zero-Phase BP PPG2", color="tab:orange", alpha=0.85)
axes[1, 0].set_title("Subplot 2a: Zero-Phase Bandpass PPG (0.5-4.0 Hz, No Startup Ringing)")
axes[1, 0].legend(loc="upper right", fontsize=8); axes[1, 0].grid(alpha=0.3)

for k, (lbl, col) in enumerate(zip(["BP Acc X", "BP Acc Y", "BP Acc Z", "BP Acc Mag"], ["tab:blue", "tab:green", "tab:purple", "black"])):
    axes[1, 1].plot(t_axis, bp_6ch[2 + k], label=lbl, color=col, alpha=0.85)
axes[1, 1].set_title("Subplot 2b: Zero-Phase Bandpass 4-Channel Accelerometer")
axes[1, 1].legend(loc="upper right", fontsize=8); axes[1, 1].grid(alpha=0.3)

# Row 2: Unit-Variance Amplitude Normalized (d1, d2 and 4-channel a[n])
axes[2, 0].plot(t_axis, d1, label="Normalized d1[n] (PPG1)", color="tab:red", alpha=0.85)
axes[2, 0].plot(t_axis, d2, label="Normalized d2[n] (PPG2)", color="tab:orange", alpha=0.85)
axes[2, 0].set_title("Subplot 3a: Unit-Variance Normalized Primary Inputs d1[n], d2[n]")
axes[2, 0].legend(loc="upper right", fontsize=8); axes[2, 0].grid(alpha=0.3)

for k, (lbl, col) in enumerate(zip(["a_x[n]", "a_y[n]", "a_z[n]", "a_mag[n]"], ["tab:blue", "tab:green", "tab:purple", "black"])):
    axes[2, 1].plot(t_axis, acc_norm_sample[k], label=lbl, color=col, alpha=0.85)
axes[2, 1].set_title("Subplot 3b: Unit-Variance Normalized 4-Channel Reference a[n]")
axes[2, 1].legend(loc="upper right", fontsize=8); axes[2, 1].grid(alpha=0.3)

# Row 3: Single-Pass MISO LMS Adaptation (Branch 1 & Branch 2)
axes[3, 0].plot(t_axis, d1, label="Primary d1[n]", color="lightgray", lw=1.1)
axes[3, 0].plot(t_axis, m_hat1_sample, label="Motion Estimate m_hat1[n]", color="tab:purple", ls="--", lw=1.2)
axes[3, 0].plot(t_axis, e1_sample, label="Cleaned e1[n] = d1 - m_hat1", color="tab:red", lw=1.3)
axes[3, 0].set_title(f"Subplot 4a: Single-Pass {opt_lms_algo} Branch 1 (M={opt_M}, SNR1={f_info['snr1']:.2f})")
axes[3, 0].legend(loc="upper right", fontsize=8); axes[3, 0].grid(alpha=0.3)

axes[3, 1].plot(t_axis, d2, label="Primary d2[n]", color="lightgray", lw=1.1)
axes[3, 1].plot(t_axis, m_hat2_sample, label="Motion Estimate m_hat2[n]", color="tab:purple", ls="--", lw=1.2)
axes[3, 1].plot(t_axis, e2_sample, label="Cleaned e2[n] = d2 - m_hat2", color="tab:orange", lw=1.3)
axes[3, 1].set_title(f"Subplot 4b: Single-Pass {opt_lms_algo} Branch 2 (M={opt_M}, SNR2={f_info['snr2']:.2f})")
axes[3, 1].legend(loc="upper right", fontsize=8); axes[3, 1].grid(alpha=0.3)

# Row 4: Late Fusion Output (Left) & Motion-Suppressed Fused Spectrum vs. Acc Spectrum (Right)
axes[4, 0].plot(t_axis, e1_sample, label=f"e1[n] (w1={f_info['w1']:.2f})", color="tab:red", alpha=0.45)
axes[4, 0].plot(t_axis, e2_sample, label=f"e2[n] (w2={f_info['w2']:.2f})", color="tab:orange", alpha=0.45)
axes[4, 0].plot(t_axis, e_fused_sample, label="Fused Signal e_fused[n]", color="black", lw=1.6)
axes[4, 0].set_title("Subplot 5a: Motion-Suppressed SNR-Weighted Late Fusion Output")
axes[4, 0].set_xlabel("Time [s]")
axes[4, 0].legend(loc="upper right", fontsize=8); axes[4, 0].grid(alpha=0.3)

axes[4, 1].plot(spec_info["freqs_bpm"], spec_info["psd_e_band"], label="Raw Fused PSD P_e(f)", color="tab:gray", alpha=0.55)
axes[4, 1].plot(spec_info["freqs_bpm"], spec_info["psd_acc_mean"], label="Mean 4-Ch Acc PSD P_acc(f)", color="tab:purple", ls="--", alpha=0.75)
axes[4, 1].plot(spec_info["freqs_bpm"], spec_info["score"], label="Motion-Suppressed Harmonic Score", color="tab:blue", lw=1.6)
axes[4, 1].axvline(true_bpm_sample, color="black", ls="--", lw=1.5, label=f"Ground-Truth ({true_bpm_sample:.1f} BPM)")
axes[4, 1].axvline(est_bpm_sample, color="tab:red", ls=":", lw=1.8, label=f"Estimated ({est_bpm_sample:.1f} BPM)")
axes[4, 1].set_title("Subplot 5b: Motion-Suppressed Fused Spectrum vs. Accelerometer Spectrum")
axes[4, 1].set_xlabel("Heart Rate Frequency [BPM]")
axes[4, 1].legend(loc="upper right", fontsize=8); axes[4, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 6. Evaluation on the Held-Out Test Dataset (`IEEEPPG_TEST.ts`)

We now run `best_model.predict_with_signals(X_test)` across all individual windows of `IEEEPPG_TEST.ts` and report pure Test AAE (`np.mean(np.abs(y_test_pred - y_test))`), Test RMSE, Composite Objective Score, and Outlier Fraction, followed by the test tracking and scatter plots.

In [ ]:
# Single-pass prediction on X_test returning both BPM estimates and cleaned window signals
y_test_pred, e_test_cleaned = best_model.predict_with_signals(X_test)

# Pure Test Set AAE, RMSE, Composite Objective Score, and Outlier Fraction
test_abs_err = np.abs(y_test_pred - y_test)
test_aae = float(np.mean(test_abs_err))
test_rmse = float(np.sqrt(np.mean(test_abs_err ** 2)))
test_obj_score = objective_function(y_test_pred, y_test)
outlier_mask = test_abs_err > 10.0
outlier_fraction = float(np.mean(outlier_mask))

print("=== Final Held-Out Test Set Summary ===")
print(f"Optimal LMS Algorithm    : {best_hyperparameters['lms_algorithm']}")
print(f"Optimal HR Detector      : {best_hyperparameters['hr_algorithm']}")
print(f"Pure Test Set AAE        : {test_aae:.3f} BPM")
print(f"Test Set RMSE            : {test_rmse:.3f} BPM")
print(f"Composite Objective Score: {test_obj_score:.3f}")
print(f"Outlier Fraction (>10BPM): {outlier_fraction:.4f} ({100.0 * outlier_fraction:.2f}% | {int(np.sum(outlier_mask))}/{len(y_test)} windows)")

# Test Set Heart Rate Tracking Trace & Overall Scatter Plot
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))

axes[0].plot(y_test, label="Ground-Truth BPM", color="black", lw=1.2)
axes[0].plot(y_test_pred, label="Estimated BPM", color="tab:blue", alpha=0.75, lw=1.0)
axes[0].set_title(f"Test Set Heart Rate Tracking (Pure AAE = {test_aae:.2f} BPM, RMSE = {test_rmse:.2f} BPM)")
axes[0].set_xlabel("Test Window Index")
axes[0].set_ylabel("Heart Rate [BPM]")
axes[0].legend()
axes[0].grid(alpha=0.3)

lims = [min(y_test.min(), y_test_pred.min()) - 5, max(y_test.max(), y_test_pred.max()) + 5]
axes[1].scatter(y_test, y_test_pred, alpha=0.35, s=15, color="tab:blue")
axes[1].plot(lims, lims, "--", color="tab:red", lw=1.2, label="Ideal (y = x)")
axes[1].set_xlim(lims)
axes[1].set_ylim(lims)
axes[1].set_title("Estimated vs. Ground-Truth BPM (Test Set)")
axes[1].set_xlabel("Ground-Truth BPM")
axes[1].set_ylabel("Estimated BPM")
axes[1].legend()
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Diagram of the Recombined Signals Before and After Pipeline Processing

Consecutive 8-second windows from the same subject advance by `STEP = 250` samples (2 seconds). By concatenating the first window with the non-overlapping 250-sample stride of subsequent windows from Subject 0, we inspect the continuous recombined waveform **before** adaptive LMS filtering/fusion and **after** pipeline reconstruction.

In [ ]:
# Recombine consecutive windows from Subject 0 (first 6 consecutive windows = 18 seconds)
subj0_indices = np.where(test_groups == 0)[0][:6]

# Before pipeline: normalized zero-phase bandpass PPG mean per window
before_windows = [
    np.mean(prepare_channels(X_test[idx])[0], axis=0)
    for idx in subj0_indices
]
# After pipeline: cleaned & motion-suppressed SNR-fused cardiac signal per window
after_windows = [e_test_cleaned[idx] for idx in subj0_indices]

recombined_before = np.concatenate([before_windows[0]] + [w[-STEP:] for w in before_windows[1:]])
recombined_after = np.concatenate([after_windows[0]] + [w[-STEP:] for w in after_windows[1:]])
t_recomb = np.arange(len(recombined_before)) / FS

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

axes[0].plot(t_recomb, recombined_before, color="tab:gray", lw=1.1, label="Recombined Signal BEFORE MISO LMS + Late Fusion")
for b in range(1, len(subj0_indices)):
    axes[0].axvline((WIN + (b - 1) * STEP) / FS, color="tab:red", ls=":", alpha=0.4)
axes[0].set_title("Recombined Multi-Window Signal BEFORE Adaptive Filtering & Late Fusion")
axes[0].set_ylabel("Normalized Amplitude")
axes[0].legend(loc="upper right"); axes[0].grid(alpha=0.3)

axes[1].plot(t_recomb, recombined_after, color="tab:blue", lw=1.2, label="Recombined Signal AFTER MISO LMS + SNR Late Fusion")
for b in range(1, len(subj0_indices)):
    axes[1].axvline((WIN + (b - 1) * STEP) / FS, color="tab:red", ls=":", alpha=0.4, label="Window Stitch Boundary" if b == 1 else None)
axes[1].set_title("Recombined Multi-Window Cardiac Signal AFTER Adaptive MISO LMS & Late Fusion")
axes[1].set_xlabel("Time [s]")
axes[1].set_ylabel("Cleaned Cardiac Signal")
axes[1].legend(loc="upper right"); axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Score by Person & Subject-Colored Scatterplot (Estimated vs. Ground-Truth BPM)

Here we break down the test performance **by person (subject ID)** and plot the **Estimated vs. Ground-Truth BPM** scatterplot where all windows belonging to the same subject share the same color.

In [ ]:
unique_test_subjects = np.unique(test_groups)
subject_metrics = []

print(f"{'Subject ID':>10s} | {'Windows':>8s} | {'Objective Score':>15s} | {'AAE [BPM]':>10s} | {'RMSE [BPM]':>10s} | {'Outliers (>10 BPM)':>18s}")
print("-" * 84)

for sid in unique_test_subjects:
    mask = test_groups == sid
    preds_s = y_test_pred[mask]
    true_s = y_test[mask]
    err_s = np.abs(preds_s - true_s)

    obj_s = objective_function(preds_s, true_s)
    aae_s = float(np.mean(err_s))
    rmse_s = float(np.sqrt(np.mean(err_s ** 2)))
    out_frac_s = float(np.mean(err_s > 10.0))

    subject_metrics.append({
        "subject_id": int(sid),
        "n_windows": int(np.sum(mask)),
        "objective": obj_s,
        "aae": aae_s,
        "rmse": rmse_s,
        "outlier_frac": out_frac_s,
    })
    print(f"Subject {int(sid):2d} | {int(np.sum(mask)):8d} | {obj_s:15.3f} | {aae_s:10.3f} | {rmse_s:10.3f} | {100.0 * out_frac_s:16.2f} %")

fig, axes = plt.subplots(1, 2, figsize=(15, 5.5))
cmap = plt.get_cmap("tab10")

# Left: Scatterplot of Estimated vs. Ground-Truth BPM colored by Subject
for idx, sid in enumerate(unique_test_subjects):
    mask = test_groups == sid
    axes[0].scatter(
        y_test[mask],
        y_test_pred[mask],
        color=cmap(idx % 10),
        alpha=0.65,
        s=22,
        label=f"Subject {int(sid)}",
    )

lims = [min(y_test.min(), y_test_pred.min()) - 5, max(y_test.max(), y_test_pred.max()) + 5]
axes[0].plot(lims, lims, "--", color="black", lw=1.4, label="Ideal (y = x)")
axes[0].plot(lims, [lims[0] + 10, lims[1] + 10], ":", color="tab:red", lw=1.0, label="+/- 10 BPM Outlier Bound")
axes[0].plot(lims, [lims[0] - 10, lims[1] - 10], ":", color="tab:red", lw=1.0)
axes[0].set_xlim(lims)
axes[0].set_ylim(lims)
axes[0].set_title("Estimated vs. Ground-Truth BPM Colored by Subject (Test Set)")
axes[0].set_xlabel("Ground-Truth BPM")
axes[0].set_ylabel("Estimated BPM")
axes[0].legend(loc="upper left", fontsize=8, ncol=2)
axes[0].grid(alpha=0.3)

# Right: Score by Person (Composite Objective Score & AAE per Subject)
subj_labels = [f"Subj {m['subject_id']}" for m in subject_metrics]
subj_obj = [m["objective"] for m in subject_metrics]
subj_aae = [m["aae"] for m in subject_metrics]
x_s = np.arange(len(subj_labels))
bar_w = 0.38

axes[1].bar(x_s - bar_w / 2, subj_obj, bar_w, label="Objective Score", color="tab:red", edgecolor="black", alpha=0.85)
axes[1].bar(x_s + bar_w / 2, subj_aae, bar_w, label="AAE [BPM]", color="tab:blue", edgecolor="black", alpha=0.85)
axes[1].set_xticks(x_s)
axes[1].set_xticklabels(subj_labels, rotation=25, ha="right")
axes[1].set_title("Test Set Error Score by Person (Subject-Wise Breakdown)")
axes[1].set_ylabel("Error Score [BPM]")
axes[1].legend()
axes[1].grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.show()

## 9. Cleaned Signal Inspection: Well-Detected Window vs. Outlier Window

Finally, we compare the cleaned cardiac waveform $e_{\text{fused}}[n]$ and its motion-suppressed spectrum for:
- A **well-detected window** ($|\hat{y}_w - y_w| \approx 0\text{ BPM}$), where the adaptive MISO LMS filter cleanly isolates the cardiac fundamental frequency.
- An **outlier window** ($|\hat{y}_w - y_w| > 10\text{ BPM}$), illustrating how severe motion artifacts or spectral ambiguity affect the signal.

In [ ]:
good_idx = int(np.argmin(test_abs_err))
outlier_indices = np.where(outlier_mask)[0]
outlier_idx = int(outlier_indices[np.argmax(test_abs_err[outlier_indices])]) if len(outlier_indices) > 0 else int(np.argmax(test_abs_err))

print(f"Total Test Windows     : {len(y_test)}")
print(f"Outlier Windows (>10)  : {int(np.sum(outlier_mask))} ({100.0 * outlier_fraction:.2f}%)")
print(f"Good Window #{good_idx:4d}      : True = {y_test[good_idx]:.2f} BPM | Est = {y_test_pred[good_idx]:.2f} BPM | Error = {test_abs_err[good_idx]:.2f} BPM")
print(f"Outlier Window #{outlier_idx:4d}   : True = {y_test[outlier_idx]:.2f} BPM | Est = {y_test_pred[outlier_idx]:.2f} BPM | Error = {test_abs_err[outlier_idx]:.2f} BPM")

_, acc_good = prepare_channels(X_test[good_idx], fs=FS)
_, acc_out = prepare_channels(X_test[outlier_idx], fs=FS)
_, spec_good = motion_suppressed_periodogram_bpm(e_test_cleaned[good_idx], acc_norm=acc_good, fs=FS, return_spectrum=True, **hr_params_spec)
_, spec_out = motion_suppressed_periodogram_bpm(e_test_cleaned[outlier_idx], acc_norm=acc_out, fs=FS, return_spectrum=True, **hr_params_spec)

fig, axes = plt.subplots(2, 2, figsize=(15, 8))

# Top-Left: Good Window Cleaned Time-Domain Signal
axes[0, 0].plot(t_axis, e_test_cleaned[good_idx], color="tab:green", lw=1.3)
axes[0, 0].set_title(f"Good Detected Window #{good_idx} — Cleaned Signal (Err = {test_abs_err[good_idx]:.2f} BPM)")
axes[0, 0].set_xlabel("Time [s]"); axes[0, 0].set_ylabel("Amplitude")
axes[0, 0].grid(alpha=0.3)

# Top-Right: Outlier Window Cleaned Time-Domain Signal
axes[0, 1].plot(t_axis, e_test_cleaned[outlier_idx], color="tab:red", lw=1.3)
axes[0, 1].set_title(f"Outlier Window #{outlier_idx} — Cleaned Signal (Err = {test_abs_err[outlier_idx]:.2f} BPM)")
axes[0, 1].set_xlabel("Time [s]"); axes[0, 1].set_ylabel("Amplitude")
axes[0, 1].grid(alpha=0.3)

# Bottom-Left: Good Window Spectrum
axes[1, 0].plot(spec_good["freqs_bpm"], spec_good["psd_e_band"], color="lightgray", label="Raw Fused PSD")
axes[1, 0].plot(spec_good["freqs_bpm"], spec_good["score"], color="tab:green", lw=1.5, label="Motion-Suppressed Score")
axes[1, 0].axvline(y_test[good_idx], color="black", ls="--", lw=1.5, label=f"True BPM ({y_test[good_idx]:.1f})")
axes[1, 0].axvline(y_test_pred[good_idx], color="tab:blue", ls=":", lw=1.8, label=f"Estimated BPM ({y_test_pred[good_idx]:.1f})")
axes[1, 0].set_title(f"Good Window #{good_idx} — Cardiac Band Spectrum")
axes[1, 0].set_xlabel("Frequency [BPM]"); axes[1, 0].set_ylabel("Normalized Score")
axes[1, 0].legend(); axes[1, 0].grid(alpha=0.3)

# Bottom-Right: Outlier Window Spectrum
axes[1, 1].plot(spec_out["freqs_bpm"], spec_out["psd_e_band"], color="lightgray", label="Raw Fused PSD")
axes[1, 1].plot(spec_out["freqs_bpm"], spec_out["score"], color="tab:red", lw=1.5, label="Motion-Suppressed Score")
axes[1, 1].axvline(y_test[outlier_idx], color="black", ls="--", lw=1.5, label=f"True BPM ({y_test[outlier_idx]:.1f})")
axes[1, 1].axvline(y_test_pred[outlier_idx], color="tab:blue", ls=":", lw=1.8, label=f"Estimated BPM ({y_test_pred[outlier_idx]:.1f})")
axes[1, 1].set_title(f"Outlier Window #{outlier_idx} — Cardiac Band Spectrum")
axes[1, 1].set_xlabel("Frequency [BPM]"); axes[1, 1].set_ylabel("Normalized Score")
axes[1, 1].legend(); axes[1, 1].grid(alpha=0.3)

plt.tight_layout()
plt.show()